# Sentiment Analysis of Newspaper Articles (NRC Emotion Lexicon)

This notebook builds on the article extraction (`articles_extracted.csv`
or an Excel file in the same format: columns `pdf_page`, `year`,
`month`, `word_count`, `text`, optionally `source_file`/`lang`/
`has_agency_marker`) and performs a **document-level sentiment
analysis using the NRC Emotion Lexicon**.

**Pipeline:**

1. Load articles (Excel or CSV)
2. Ensure language per article (German/English -- if the column
   `lang` is missing, it is determined here)
3. Load the NRC lexicon -- **German and English**, from the official
   multilingual NRC file (Google Translate translations of the
   original English words)
4. Umlaut transliteration of the German lexicon (`für` -> `fuer` etc.,
   as used in historical newspaper typesetting)
5. Tokenization/lemmatization per article (spaCy, language-dependent)
6. Document-level scoring: relative frequency per emotion, normalized
   by text length
7. Aggregation by year + visualization of the sentiment trend
8. Ranking articles by emotion/year -> close reading
9. Coverage check (what percentage of words even match in the
   lexicon?) -- important for interpreting the results


## 0. Setup

Requires: `pandas`, `openpyxl` (Excel), `spacy` with the models
`de_core_news_sm` and `en_core_web_sm`, `langdetect`, `matplotlib`,
`rapidfuzz` (fuzzy matching, Section 5).

```bash
pip install pandas openpyxl spacy langdetect matplotlib rapidfuzz compound-split scipy statsmodels
python -m spacy download de_core_news_sm
python -m spacy download en_core_web_sm
```


In [ ]:
#!pip install pandas openpyxl spacy langdetect matplotlib rapidfuzz compound-split scipy statsmodels
#!python -m spacy download de_core_news_sm
#!python -m spacy download en_core_web_sm

import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import spacy
import os
import glob
from rapidfuzz import process, fuzz
from compound_split import char_split
from scipy import stats
from statsmodels.stats.multitest import multipletests
from langdetect import detect, DetectorFactory

DetectorFactory.seed = 0
warnings.filterwarnings("ignore")

print("Loading spaCy models (German + English) ...")
nlp_de = spacy.load("de_core_news_sm")
nlp_en = spacy.load("en_core_web_sm")
print("Done.")


## 1. Load articles

Set `ARTICLES_PATH` to your file. Works with both `.xlsx`
and `.csv` (automatically detected from the file extension).


In [ ]:
ARTICLES_PATH = "articles_extracted.csv"  # <-- adjust to your .xlsx if needed

def load_articles(path):
    path = Path(path)
    if path.suffix.lower() in (".xlsx", ".xls"):
        df = pd.read_excel(path)
    else:
        df = pd.read_csv(path)
    return df

articles = load_articles(ARTICLES_PATH)
print(f"{len(articles)} articles loaded.")
print("Columns:", list(articles.columns))
articles.head(3)


## 1b. Data cleaning

Three issues that showed up on a real, larger extraction run and
should be fixed before the sentiment analysis:

**a) Year plausibility filter + fill-forward.** OCR occasionally misreads
the year in the masthead (e.g. "1194" or "7941" instead of "1941"). Implausible
values (outside the known publication period) are removed and
filled in from neighboring pages of the **same file** (issues span
multiple consecutive pages) -- adjust `YEAR_MIN`/`YEAR_MAX` if the
corpus covers a different period.

**b) Filter out OCR noise/garbage.** Some blocks from very densely set
advertisement pages consist almost entirely of dashes/symbols instead of real text
(e.g. `"— — — al — — na — ="`) and slip through the advertisement filter
because they are technically "long enough". One telltale sign is that
`langdetect` misclassifies them as an exotic language (French,
Welsh, ...) -- itself a hint of junk data, but
here they are filtered directly via the proportion of non-alphabetic "words".

**c) Remove leading digits.** Occasionally an advertisement
reference number is attached at the start of an article (e.g. `"52825 Aktionen gegen..."`) --
presumably leaked in from a neighboring block.


In [ ]:
YEAR_MIN, YEAR_MAX = 1938, 1946  # known publication period -- adjust!
JUNK_RATIO_THRESHOLD = 0.3        # proportion of non-alphabetic "words"

def clean_leading_number(text):
    return re.sub(r"^\d+\s+", "", text)

def junk_ratio(text):
    tokens = text.split()
    if not tokens:
        return 1.0
    non_alpha = sum(1 for t in tokens if not re.search(r"[a-zA-Z\u00e4\u00f6\u00fc\u00c4\u00d6\u00dc\u00df]{2,}", t))
    return non_alpha / len(tokens)


n_before = len(articles)

# a) Year plausibility filter + fill-forward per source file
if "year" in articles.columns:
    articles = articles.sort_values(
        [c for c in ["source_file", "pdf_page"] if c in articles.columns]
    ).reset_index(drop=True)
    valid_year = articles["year"].between(YEAR_MIN, YEAR_MAX)
    n_invalid = (~valid_year & articles["year"].notna()).sum()
    articles.loc[~valid_year, "year"] = None
    group_col = "source_file" if "source_file" in articles.columns else None
    if group_col:
        articles["year"] = articles.groupby(group_col)["year"].transform(lambda s: s.ffill().bfill())
    else:
        articles["year"] = articles["year"].ffill().bfill()
    print(f"Implausible years removed: {n_invalid}")
    print(f"Missing years after fill-forward: {articles['year'].isna().sum()}")

# b) Filter out OCR noise/garbage
articles["junk_ratio"] = articles["text"].apply(junk_ratio)
n_junk = (articles["junk_ratio"] > JUNK_RATIO_THRESHOLD).sum()
articles = articles[articles["junk_ratio"] <= JUNK_RATIO_THRESHOLD].drop(columns=["junk_ratio"]).reset_index(drop=True)
print(f"Removed as OCR noise: {n_junk}")

# c) Remove leading digits
articles["text"] = articles["text"].apply(clean_leading_number)

print(f"\n{n_before} -> {len(articles)} articles after cleaning.")


## 2. Ensure language per article

If the `lang` column already exists (from the extraction pipeline),
it is used as-is. If not, it is determined here.
This matters because the newspaper appears as *"English and German
Editions"* -- German and English each need their own NRC lexicon.


In [ ]:
def detect_language(text):
    if not isinstance(text, str) or len(text.split()) < 5:
        return "unknown"
    try:
        return detect(text)
    except Exception:
        return "unknown"

if "lang" not in articles.columns:
    print("No 'lang' column found -- detecting language per article ...")
    articles["lang"] = articles["text"].apply(detect_language)

print(articles["lang"].value_counts())


## 3. Load the NRC lexicon (German + English)

The official NRC Emotion Lexicon website (saifmohammad.com) requires a
short license form for the multilingual version (research use).
If the file `NRC-Emotion-Lexicon-v0.92-InManyLanguages-web.xlsx`
is not yet available: download it there (accept the license terms,
research use is the standard case) and place it in the project folder.

**Column layout of the file** (as of this version): column 0 = English
word, column 12 = German translation, columns 41-50 = the ten
NRC categories (Positive, Negative, Anger, Anticipation, Disgust, Fear,
Joy, Sadness, Surprise, Trust) as 0/1 flags. The code below looks up
columns by **name** rather than fixed indices -- more robust in case
the column order differs slightly in a given version.


In [ ]:
NRC_PATH = "NRC-Emotion-Lexicon-v0.92-InManyLanguages-web.xlsx"

EMOTIONS = ["Positive", "Negative", "Anger", "Anticipation", "Disgust",
            "Fear", "Joy", "Sadness", "Surprise", "Trust"]


def load_nrc_lexicon(path, emotions=EMOTIONS):
    """Loads the multilingual NRC file and builds two lexicons:
    English (word -> set of emotions) and German (translated
    word -> set of emotions)."""
    df = pd.read_excel(path)

    # Find columns by name rather than fixed indices -- more robust
    # against slightly different file versions.
    en_col = "English Word"
    de_col = [c for c in df.columns if c.startswith("German")][0]
    emotion_cols = {e: e for e in emotions if e in df.columns}
    missing = set(emotions) - set(emotion_cols)
    if missing:
        raise ValueError(f"Emotion columns not found: {missing}. "
                          f"Available columns: {list(df.columns)}")

    lex_en, lex_de = {}, {}
    for _, row in df.iterrows():
        active_emotions = {e for e in emotions if row.get(e) == 1}
        if not active_emotions:
            continue  # skip words with no emotion assignment at all

        en_word = str(row[en_col]).strip().lower()
        if en_word and en_word != "nan":
            lex_en.setdefault(en_word, set()).update(active_emotions)

        de_word = row[de_col]
        if isinstance(de_word, str) and de_word.strip():
            de_word = de_word.strip().lower()
            lex_de.setdefault(de_word, set()).update(active_emotions)

    return lex_en, lex_de


lex_en, lex_de = load_nrc_lexicon(NRC_PATH)
print(f"English lexicon: {len(lex_en)} words")
print(f"German lexicon (before transliteration): {len(lex_de)} words")


## 4. Umlaut transliteration of the German lexicon

Historical newspaper typesetting consistently uses `ae/oe/ue/ss` instead of
`ä/ö/ü/ß` (no umlaut was available in the letterpress/telegraphy
character set of the time) -- not an OCR error, but how the text was
actually printed. The lexicon is extended with transliterated
additional entries (`für` -> also `fuer`), so that both spellings match.


In [ ]:
_UMLAUT_MAP = str.maketrans({"ä": "ae", "ö": "oe", "ü": "ue"})


def delatinize(word):
    word = word.replace("ß", "ss")
    return word.translate(_UMLAUT_MAP)


def delatinize_lexicon(word2emotions):
    extended = dict(word2emotions)
    for word, emotions in word2emotions.items():
        alt = delatinize(word)
        if alt != word:
            extended[alt] = extended.get(alt, set()) | emotions
    return extended


lex_de = delatinize_lexicon(lex_de)
print(f"German lexicon (after transliteration): {len(lex_de)} words")

# Quick test:
print("'fuer' in the lexicon:", "fuer" in lex_de, "->", lex_de.get("fuer"))


## 5. Tokenization/lemmatization

Per article: lemmatize the text (language-dependent spaCy model). **Important
fix:** spaCy's German model capitalizes noun lemmas even for lowercase
input (`polizei` -> `Polizei`), but the lexicon is consistently
lowercase -- without `.lower()` on the lemma, coverage would be
unnecessarily lost.

**In addition: POS filter.** Beyond `is_stop`, word classes that never
contribute anything in the NRC lexicon anyway are explicitly excluded
(pronouns, articles, auxiliary verbs, numbers, conjunctions -- spaCy's
stopword list does not catch all of these). This only gains one or two
percentage points, but is entirely risk-free (it only removes words
that would never match anyway) and cleanly shrinks the denominator of
the coverage calculation.

**In addition: a cross-language stopword filter -- with a safety net.**
The newspaper is bilingual, and some extracted "articles" actually
contain mixed-language text (e.g. when two adjacent letters to the editor
-- one German, one English -- were merged into one block during
extraction). The language model used for an article does not recognize
the stopwords of the OTHER language (`the` remains an ordinary word for
the German model, `der` for the English model) -- so foreign-language
function words slip through unfiltered.

**Important -- a naive union of both stopword lists would be a mistake
here:** some German stopwords happen to also be real, strongly
emotion-laden ENGLISH words -- most notably `war` (Krieg!) and `die`
(sterben!), as well as `gut` (English "guts/nerve"), `wart`/`wen`
(medical terms). A blind combined list would filter `war`/`die` out of
EVERY English article -- for a newspaper from wartime, a noticeable
loss of information, not an edge case. Therefore, a foreign-language
stopword is only filtered if it does NOT happen to be in the emotion
lexicon of the article's actual language:


In [ ]:
from spacy.lang.de.stop_words import STOP_WORDS as DE_STOP_WORDS
from spacy.lang.en.stop_words import STOP_WORDS as EN_STOP_WORDS

CONTENT_POS = {"NOUN", "PROPN", "VERB", "ADJ", "ADV"}

def lemmatize(text, lang):
    nlp = nlp_de if lang == "de" else nlp_en
    primary_lexicon = lex_de if lang == "de" else lex_en
    cross_lang_stop = EN_STOP_WORDS if lang == "de" else DE_STOP_WORDS  # the OTHER language

    doc = nlp(str(text).lower())
    result = []
    for tok in doc:
        if not tok.is_alpha or tok.is_stop or tok.pos_ not in CONTENT_POS:
            continue
        lemma = tok.lemma_.lower()
        # Only filter out a foreign-language stopword if it is not
        # coincidentally a genuine emotion word of the article's actual
        # language (e.g. "war"/"die" are German stopwords, but strongly
        # emotion-laden words in English -- those should stay).
        if lemma in cross_lang_stop and lemma not in primary_lexicon:
            continue
        result.append((lemma, tok.pos_))
    return result


articles["_lang_used"] = articles["lang"].where(articles["lang"].isin(["de", "en"]), "de")

print("Lemmatizing articles (may take a few minutes depending on the count) ...")
articles["_lemmas_pos"] = [
    lemmatize(text, lang) for text, lang in zip(articles["text"], articles["_lang_used"])
]
print("Done.")


**If mixed-language articles themselves matter for the analysis** (not
just the stopwords in the topic lists, but a correct emotion
assignment for BOTH language portions of a mixed article): this would
need to be addressed in the extraction pipeline (`extract_articles.py`)
-- e.g. by searching for language switches within an article block
(a rough approach: separate language detection per line/sentence
instead of once per whole article) and splitting the block into two
separate articles at that point. Considerably more effort than the fix
here, and for a few dozen affected articles out of ~1000, probably not
the best investment -- the current fix already fully addresses the
visible symptom in the topic lists.


## 5b. Fuzzy-matching correction table

Even after the lowercase fix, NRC coverage remains noticeably below
that of English -- OCR typos and slightly different word forms (e.g.
`"zwischenfalle"` instead of `"zwischenfall"`) do not match exactly.
Fuzzy matching (via `rapidfuzz`) addresses this.

**Important for performance:** matching is NOT redone per article;
instead, all **unique**, not-yet-matched words across the whole corpus
are collected once, matched against the lexicon once, and a
correction table (word -> lexicon word) is built from that. Only then
is scoring performed. This is many times faster than fuzzy matching
directly inside the scoring loop.

`FUZZY_SCORE_CUTOFF = 88` was deliberately chosen conservatively (tested
on a sample: at 88, matches are almost consistently sensible variants
of the same word, e.g. number/case forms; below that, the risk of
confusions such as `"Bestelle"` -> `"stelle"` increases).


In [ ]:
FUZZY_SCORE_CUTOFF = 88
MIN_WORD_LEN_FOR_FUZZY = 4  # very short words produce too many random matches

def build_fuzzy_corrections(unmatched_words, lexicon, score_cutoff=FUZZY_SCORE_CUTOFF):
    lexicon_words = list(lexicon.keys())
    corrections = {}
    for w in unmatched_words:
        if len(w) < MIN_WORD_LEN_FOR_FUZZY:
            continue
        match = process.extractOne(w, lexicon_words, scorer=fuzz.ratio, score_cutoff=score_cutoff)
        if match:
            corrections[w] = match[0]
    return corrections


def collect_unmatched(df, lang_value, lexicon):
    words = set()
    for lemmas_pos, lang in zip(df["_lemmas_pos"], df["_lang_used"]):
        if lang == lang_value:
            words.update(w for w, pos in lemmas_pos if w not in lexicon)
    return words


print("Building fuzzy correction table (German) ...")
unmatched_de = collect_unmatched(articles, "de", lex_de)
fuzzy_de = build_fuzzy_corrections(unmatched_de, lex_de)
print(f"  {len(fuzzy_de)} of {len(unmatched_de)} unique unmatched words corrected.")

print("Building fuzzy correction table (English) ...")
unmatched_en = collect_unmatched(articles, "en", lex_en)
fuzzy_en = build_fuzzy_corrections(unmatched_en, lex_en)
print(f"  {len(fuzzy_en)} of {len(unmatched_en)} unique unmatched words corrected.")

FUZZY_MAP = {"de": fuzzy_de, "en": fuzzy_en}


## 5b2. Compound splitting (German only)

German compound words (`Kriegsversorgung`, `Weltkrieg`, `Kulturleben`, ...)
almost never match the NRC lexicon as a whole, even though their
components (`Versorgung`, `Krieg`, `Kultur`) are indeed listed.
`compound-split` (CharSplit algorithm) is used to tentatively split the
word into two parts and check whether one of the parts matches.

**Note: higher risk of error than with fuzzy matching!** The algorithm
occasionally splits proper nouns in misleading ways (`Stanislaus` ->
`Stanis`+`Laus`, would falsely introduce "Disgust"; `Entscheidung` ->
`Ent`+`Scheidung`, would falsely associate with "Scheidung"/divorce and
sadness). This risk is limited by two rules, calibrated on a sample:

- **Only words with POS tag `NOUN`** (not `PROPN` -- excludes most
  proper nouns/place names, the most common error source)
- **Only splits with score > 0.7** (CharSplit's own confidence --
  filters out most nonsensical splits)

This reduces the number of matches compared to a looser setting (on the
sample: 63 instead of 201 additional matches), but the remaining matches
are almost consistently genuine, sensible compounds.


In [ ]:
COMPOUND_SCORE_CUTOFF = 0.7
MIN_WORD_LEN_FOR_COMPOUND = 8

def build_compound_corrections(df, lexicon, fuzzy_map, score_cutoff=COMPOUND_SCORE_CUTOFF):
    """Builds a correction table ONLY for German NOUN lemmas that
    matched neither exactly nor via fuzzy matching."""
    candidates = set()
    for lemmas_pos, lang in zip(df["_lemmas_pos"], df["_lang_used"]):
        if lang != "de":
            continue
        for w, pos in lemmas_pos:
            if (pos == "NOUN" and len(w) >= MIN_WORD_LEN_FOR_COMPOUND
                    and w not in lexicon and w not in fuzzy_map):
                candidates.add(w)

    corrections = {}
    for w in candidates:
        splits = char_split.split_compound(w)
        if not splits:
            continue
        score, part1, part2 = splits[0]
        if score <= score_cutoff:
            continue
        p1, p2 = part1.lower(), part2.lower()
        # Prefer the part that matches exactly; otherwise the one corrected via fuzzy matching
        for part in (p1, p2):
            if part in lexicon:
                corrections[w] = part
                break
        else:
            for part in (p1, p2):
                if part in fuzzy_map:
                    corrections[w] = fuzzy_map[part]
                    break
    return corrections


print("Building compound-splitting correction table (German only, NOUN, score>0.7) ...")
compound_de = build_compound_corrections(articles, lex_de, fuzzy_de)
print(f"  {len(compound_de)} additional compound matches.")

# Merged into the existing fuzzy map -- score_document does not need to be
# adjusted separately, compound matches are treated like fuzzy matches.
FUZZY_MAP["de"] = {**fuzzy_de, **compound_de}


## 5c. Document-level scoring


In [ ]:
def score_document(lemmas, lexicon, fuzzy_map, emotions=EMOTIONS):
    """Two variants per emotion:
    - Raw score (e.g. "Joy"): share of ALL words in the article.
      Intuitively readable as "overall intensity", BUT: depends on
      NRC coverage. With lower coverage (e.g. German vs. English,
      or later years with more OCR noise), ALL emotion categories
      shrink equally -- a comparison across languages/years with
      different coverage is thereby distorted.
    - Coverage-normalized score (suffix "_norm", e.g. "Joy_norm"):
      share of the words THAT WERE ACTUALLY MATCHED. Removes the
      coverage effect -- a fairer comparison across languages/years
      with different lexicon match rates.
    """
    counts = {e: 0 for e in emotions}
    matched = 0
    for word in lemmas:
        word_emotions = lexicon.get(word)
        if word_emotions is None:
            corrected = fuzzy_map.get(word)
            if corrected:
                word_emotions = lexicon.get(corrected)
        if word_emotions:
            matched += 1
            for e in word_emotions:
                counts[e] += 1
    total = max(len(lemmas), 1)
    scores = {e: counts[e] / total for e in emotions}
    scores["nrc_coverage"] = matched / total  # see Section 9
    matched_safe = max(matched, 1)  # avoid division by zero with 0 matches
    for e in emotions:
        scores[f"{e}_norm"] = counts[e] / matched_safe
    return scores


def process_article(row):
    lang = row["_lang_used"]
    lexicon = lex_de if lang == "de" else lex_en
    words = [w for w, pos in row["_lemmas_pos"]]
    return score_document(words, lexicon, FUZZY_MAP[lang])


print("Scoring articles ...")
scores = articles.apply(process_article, axis=1, result_type="expand")
articles = pd.concat([articles, scores], axis=1)
# Note: "_lemmas_pos" and "_lang_used" are deliberately kept (not
# dropped) -- Section 11 (topic analysis) reuses them without
# re-lemmatizing. They are only removed when the final results are saved.
print("Done.")
articles[["year", "lang", "word_count", "Positive", "Negative", "Joy",
          "Anger", "nrc_coverage"]].head(10)


## 6. Aggregation by year + visualization

Average emotion shares per year -- both raw (`Positive`, ...)
and coverage-normalized (`Positive_norm`, ...). **Comparing the two
variants shows how much of the yearly trend is "real" and how much
just reflects declining NRC coverage** (see Section 9):
if a curve drops sharply in the raw plot but stays flat in the
normalized plot, the apparent trend was largely a coverage artifact.


In [ ]:
NORM_EMOTIONS = [f"{e}_norm" for e in EMOTIONS]
yearly = articles.groupby("year")[EMOTIONS + NORM_EMOTIONS + ["nrc_coverage"]].mean().reset_index()
yearly


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Top: raw -- positive/negative and basic emotions
axes[0, 0].plot(yearly["year"], yearly["Positive"], marker="o", label="Positive", color="#2a9d5c")
axes[0, 0].plot(yearly["year"], yearly["Negative"], marker="o", label="Negative", color="#c0392b")
axes[0, 0].set_title("Positive vs. Negative per year (non-normalised)")
axes[0, 0].set_ylabel("Relative Frequency")
axes[0, 0].legend()

basic = ["Anger", "Anticipation", "Disgust", "Fear", "Joy", "Sadness", "Surprise", "Trust"]
for e in basic:
    axes[0, 1].plot(yearly["year"], yearly[e], marker=".", label=e)
axes[0, 1].set_title("NRC emotion scores per year (non-normalised)")
axes[0, 1].legend(fontsize=7, ncol=2)

# Bottom: coverage-normalised -- the same comparison
axes[1, 0].plot(yearly["year"], yearly["Positive_norm"], marker="o", label="Positive", color="#2a9d5c")
axes[1, 0].plot(yearly["year"], yearly["Negative_norm"], marker="o", label="Negative", color="#c0392b")
axes[1, 0].set_title("Positive vs. Negative per year (coverage-normalised)")
axes[1, 0].set_xlabel("Year")
axes[1, 0].set_ylabel("Percentage of matched words")
axes[1, 0].legend()

for e in basic:
    axes[1, 1].plot(yearly["year"], yearly[f"{e}_norm"], marker=".", label=e)
axes[1, 1].set_title("NRC emotion scores per year (coverage-normalised)")
axes[1, 1].set_xlabel("Year")
axes[1, 1].legend(fontsize=7, ncol=2)

plt.tight_layout()
plt.savefig("sentiment_trends.png", dpi=150)
plt.show()


In [ ]:
# For context right next to it: how much does coverage itself decline over the years?
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(yearly["year"], yearly["nrc_coverage"], marker="o", color="#555")
ax.set_title("NRC coverage per year")
ax.set_xlabel("Year")
ax.set_ylabel("Proportion of matched words")
plt.tight_layout()
plt.savefig("nrc_coverage.png", dpi=150)
plt.show()



## 7. Rank articles by emotion (close reading)

For a more detailed qualitative analysis: pull out the articles with
the highest share of a given emotion -- e.g. to specifically read the
"angriest" or "most hopeful" articles of a given year.

Here the **raw** score (`Fear`, `Joy`, ...) is usually more suitable
than the normalized one: the goal is to find articles that are
*overall* strongly emotionally charged, not just relative to their few
matched words (a very short article with 2 of 3 matched words in
"Anger" would have a very high `Anger_norm` value, despite containing
almost no text overall).


In [ ]:
def top_articles_by_emotion(df, emotion, year=None, n=10, save_csv=False, output_path="top_articles.csv"):
    subset = df if year is None else df[df["year"] == year]
    cols = ["pdf_page", "year", "lang", emotion, "text"]
    top_articles = subset.sort_values(emotion, ascending=False).head(n)[cols]

    if save_csv:
        top_articles.to_csv(output_path, index=False)
        print(f"Table saved to: {output_path}")

    return top_articles


# Example: the 10 articles with the highest fear share, across all years
top_articles_by_emotion(articles, "Fear", n=10, save_csv=True, output_path="CSV/top_fear_articles.csv")


In [ ]:
def top_articles_by_emotion(df, emotion, year=None, n=10, save_csv=False, output_path="top_articles.csv"):
    subset = df if year is None else df[df["year"] == year]
    cols = ["pdf_page", "year", "lang", emotion, "text"]
    top_articles = subset.sort_values(emotion, ascending=False).head(n)[cols]

    if save_csv:
        top_articles.to_csv(output_path, index=False)
        print(f"Table saved to: {output_path}")

    return top_articles


# Example: the 10 articles with the highest joy share, across all years
top_articles_by_emotion(articles, "Joy", n=10, save_csv=True, output_path="CSV/top_joy_articles.csv")


In [ ]:
# Example: the 5 "happiest" articles per year
top_joy_per_year = (
    articles.sort_values("Joy", ascending=False)
    .groupby("year")
    .head(5)
    .sort_values(["year", "Joy"], ascending=[True, False])
)
top_joy_per_year[["pdf_page", "year", "Joy", "text"]]

# Save table as CSV
#top_joy_per_year.to_csv("top_joy_articles_per_year.csv", index=False)
#print("Table saved to: top_joy_articles_per_year.csv")


## 8. Save results


In [ ]:
articles.drop(columns=["_lemmas_pos", "_lang_used", "source_file"], errors="ignore").to_csv(
    "articles_with_sentiment.csv", index=False
)
yearly.to_csv("sentiment_by_year.csv", index=False)
print("Saved: articles_with_sentiment.csv, sentiment_by_year.csv, sentiment_trends.png")


## 9. Coverage check -- before interpreting the results

**Important:** what percentage of an article's words even find a match
in the NRC lexicon? For historical, OCR-affected texts this can be low
(spelling variants, OCR errors, outdated word forms).

Two patterns worth watching for:

- **Different coverage between groups** (e.g. German vs.
  English, or early vs. late years): the *raw* scores of these groups
  are then not directly comparable, because one group systematically
  "loses" more words than the other. For such comparisons, use the
  `_norm` columns (Section 6 shows both variants side by side).
- **Low coverage for individual short articles**: the score of a
  60-word article with only 3 matched words is considerably less
  reliable than that of a 60-word article with 15 matched words -- when
  averaged over many articles into yearly figures this tends to even
  out, but for single-article rankings (Section 7) it is worth also
  checking `nrc_coverage`.


In [ ]:
print("Overall NRC coverage:")
print(articles["nrc_coverage"].describe())

print()
print("Coverage by language:")
print(articles.groupby("lang")["nrc_coverage"].mean())

print()
print("Coverage by year (layout/OCR quality can change over time):")
print(articles.groupby("year")["nrc_coverage"].mean())


## 10. Agency reports vs. editorial articles

Agency reports (Reuter, Transocean, Havas, D.N.B., ...) are often
stylistically more sober than editorial articles -- a direct
comparison shows whether and how strongly this is reflected in the
NRC score. Uses the `has_agency_marker` column from the extraction
pipeline.


In [ ]:
comparison = (
    articles.groupby("has_agency_marker")[EMOTIONS + NORM_EMOTIONS + ["nrc_coverage", "word_count"]]
    .mean()
    .rename(index={True: "Agency", False: "Editorial"})
)
comparison


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

x = range(len(EMOTIONS))
width = 0.35
labels_agency = comparison.index.tolist()

axes[0].bar([i - width/2 for i in x], comparison.loc[labels_agency[0], EMOTIONS], width, label=labels_agency[0])
axes[0].bar([i + width/2 for i in x], comparison.loc[labels_agency[1], EMOTIONS], width, label=labels_agency[1])
axes[0].set_xticks(list(x))
axes[0].set_xticklabels(EMOTIONS, rotation=45, ha="right")
axes[0].set_title("Agency vs. Editorial (non-normalised)")
axes[0].set_ylabel("Relative Frequency")
axes[0].legend()

norm_cols = [f"{e}_norm" for e in EMOTIONS]
axes[1].bar([i - width/2 for i in x], comparison.loc[labels_agency[0], norm_cols], width, label=labels_agency[0])
axes[1].bar([i + width/2 for i in x], comparison.loc[labels_agency[1], norm_cols], width, label=labels_agency[1])
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(EMOTIONS, rotation=45, ha="right")
axes[1].set_title("Agency vs. Editorial (normalised)")
axes[1].legend()

plt.tight_layout()
plt.savefig("sentiment_agency_comparison.png", dpi=150)
plt.show()


In [ ]:
# The same broken down by year as well -- does the agency/editorial
# difference stay stable over time, or do the two groups converge/diverge?
by_year_agency = (
    articles.groupby(["year", "has_agency_marker"])[EMOTIONS + NORM_EMOTIONS]
    .mean()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(8, 5))
for is_agency, label, color in [(True, "Agency Articles", "#c0392b"), (False, "Editorial Articles", "#2a78d6")]:
    subset = by_year_agency[by_year_agency["has_agency_marker"] == is_agency]
    ax.plot(subset["year"], subset["Negative_norm"], marker="o", label=label, color=color)
ax.set_title("Negative Words in Agency vs. Editorial Articles per year (normalised)")
ax.set_xlabel("Year")
ax.set_ylabel("Proportion of matched words")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
comparison.to_csv("sentiment_by_agency_marker.csv")
by_year_agency.to_csv("sentiment_by_year_and_agency.csv", index=False)
print("Saved: sentiment_by_agency_marker.csv, sentiment_by_year_and_agency.csv")


## 11. Thematic analysis per emotion

What content is typical for articles with a high share of a given
emotion? For each emotion, the top-N articles (by raw score) are taken
and combined into **one** "document". Using TF-IDF (compared against
the documents of the other emotions), the words that are especially
characteristic of exactly this emotion group emerge -- frequent enough
within the group, but unusually rare in the others.

**What this is and is not:** this is NOT standalone topic modelling
(like LDA) -- it does not learn independent topics, it only shows which
vocabulary distinguishes each emotion group from the others. This
makes it robust even for a corpus of this size and requires no manual
topic labelling.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

TOP_N_ARTICLES_PER_EMOTION = 30
TOP_TERMS_TO_SHOW = 15

def build_emotion_topic_words(df, emotions=EMOTIONS, top_n=TOP_N_ARTICLES_PER_EMOTION,
                               top_terms=TOP_TERMS_TO_SHOW):
    """Builds one "document" per emotion from the lemmas of the top-N
    articles (by raw score) and determines the most characteristic
    words of that group relative to the other emotion groups via
    TF-IDF."""
    emotion_docs = {}
    for e in emotions:
        top = df.sort_values(e, ascending=False).head(top_n)
        words = []
        for lemmas_pos in top["_lemmas_pos"]:
            words.extend(w for w, pos in lemmas_pos)
        emotion_docs[e] = " ".join(words)

    vec = TfidfVectorizer(max_df=0.9, min_df=1)
    tfidf = vec.fit_transform(list(emotion_docs.values()))
    terms = vec.get_feature_names_out()

    results = {}
    for i, e in enumerate(emotions):
        row = tfidf[i].toarray()[0]
        top_idx = row.argsort()[::-1][:top_terms]
        results[e] = [terms[j] for j in top_idx if row[j] > 0]
    return results


# Split by language, since each has its own vocabulary/lexicon
print("=== German ===")
topic_words_de = build_emotion_topic_words(articles[articles["_lang_used"] == "de"])
for e, words in topic_words_de.items():
    print(f"{e:15} {', '.join(words)}")


In [ ]:
print("=== English ===")
topic_words_en = build_emotion_topic_words(articles[articles["_lang_used"] == "en"],
                                            top_n=15)  # fewer articles available
for e, words in topic_words_en.items():
    print(f"{e:15} {', '.join(words)}")


In [ ]:
# Save as a table
topic_rows = []
for e, words in topic_words_de.items():
    topic_rows.append({"lang": "de", "emotion": e, "top_terms": ", ".join(words)})
for e, words in topic_words_en.items():
    topic_rows.append({"lang": "en", "emotion": e, "top_terms": ", ".join(words)})
pd.DataFrame(topic_rows).to_csv("CSV/emotion_topic_words.csv", index=False)
print("Saved: emotion_topic_words.csv")


## 12. Do specific diaspora groups/governments correlate with emotions?

For a set of historically interesting groups (Jewish diaspora
subgroups), governments/occupying powers, aid organizations, and core
themes, this checks whether articles mentioning them differ in their
emotion profile from the rest of the corpus.

**Important methodological limitations, before the numbers:**

1. **Detection via keyword search** (bilingual keyword list, full-text
   search on the raw text, with word boundaries `\b...\b`). A match
   only means "the group/institution is mentioned somewhere in the
   article", not "the article is centrally about it". This tends to
   dilute genuine effects (not: create them).
2. **Word boundaries are essential, otherwise false positives without
   end.** Without `\b...\b`, the abbreviation `IC`, for example,
   matches as a substring in practically any word containing "ic"
   ("wirklich", "politisch") -- in an initial test, this caused one
   category to jump from a plausible ~6 to an absurd 961 matches. With
   word boundaries, the problem disappears.
3. **Some search terms remained too ambiguous regardless and were
   removed:** `Mir` (Mir Yeshiva) is, in German, also the ordinary
   pronoun ("sagte mir", "zeigte mir") -- a sample check found 63 of 63
   matches were the pronoun, none the yeshiva. `ORT` (aid organization)
   collides in the same way with the German word "Ort/place" ("an
   diesem Ort") -- not reliably separable even with capitalization due
   to inconsistent OCR capitalization. `Pass` alone produced only a
   single (incorrect) match and was removed; `Passierschein`/`Permit`
   already cover the intended concept.
4. **Sample size determines whether a test is meaningful at all.**
   Categories with too few matches are not formally tested, only their
   match count is shown.
5. **Multiple-comparison correction is essential:** across many topics
   x 10 emotions, "significant" findings would otherwise arise by
   chance alone. Corrected via Benjamini-Hochberg (FDR) across all
   tests.
6. **Correlation, not causation** -- possible confounders (year, agency
   report vs. editorial, article length) are NOT controlled for here.


In [ ]:
def _kw_pattern(terms):
    """Builds a regex search pattern with word boundaries around EACH
    individual term -- prevents substring false matches such as "IC" in
    "wirklich"."""
    return "|".join(r"\b" + re.escape(t.lower()) + r"\b" for t in terms)


# Bilingual keyword lists -- each category deliberately includes both a
# German AND an English equivalent, so that articles are found
# regardless of their language (an earlier version was monolingual in
# several places, e.g. "SMC & Western powers" entirely in English).
# "Mir", "ORT", "Pass" are deliberately NOT included (see rationale
# above -- too generic in German, checked against samples: "mir" is
# exclusively the pronoun, not the Mir Yeshiva; "Ort" predominantly the
# word "place", not the ORT organization).
TOPIC_KEYWORDS = {
    # --- Diaspora groups ---
    "Central European Refugees (DE/AT)": _kw_pattern([
        "Emigranten","Emigrant","Fluechtlinge","Fluechtling","Refugees","Refugee",
        "Neulaender","Passagiere","Hongkew","Hongkou","Little Vienna","Mitteleuropaeer",
        "Central European","Reichsdeutsche","Austrians","Oesterreicher","German Jews",
        "Deutsche Juden","Heimbeobachter",
    ]),
    "Sephardi Jews": _kw_pattern([
        "Baghdadi","Bagdadi","Sephardim","Sephardic","Sephardische Juden","Sassoon",
        "Kadoorie","Ezra","Hardoon","British Jews","Britische Juden","Iraqi Jews",
        "Irakische Juden","Eastern Jews","Orientalische Juden",
    ]),
    "Russian-Ashkenazi Jews": _kw_pattern([
        "Russische Juden","Russian Jews","Russian Ashkenazi","SAJCA","French Concession",
        "Franzoesische Konzession","Frenchtown","Harbin Jews","Harbiner Juden",
        "Ashkenazi Community","Aschkenasische Gemeinde",
    ]),
    "Polish-Eastern European/Orthodox Groups": _kw_pattern([
        "Polnische Juden","Polish Jews","EASTJEWCOM","Mirer","Yeshiva","Jeschiwa",
        "Rabbiner","Rabbi","Orthodox","Kobe refugees","Kobe Fluechtlinge","Ostjuden",
        "Eastern European Jews","Litauische Juden","Lithuanian Jews",
    ]),
    # --- Governments/occupation/authorities ---
    "SMC & Western powers": _kw_pattern([
        "SMC","Shanghai Municipal Council","Munizipalrat","Municipal","Settlement",
        "International Settlement","Internationale Niederlassung","French Concession",
        "Franzoesische Konzession","Consulate","Konsulat","Consular","Konsularisch",
        "SMP","Special Branch","Ratepayers","Steuerzahler",
    ]),
    "Japanese Occupation Authorities": _kw_pattern([
        "Japanese","Japanisch","Japanese Authorities","Japanische Behoerden",
        "Japanese Military","Japanisches Militaer","Bureau for Stateless Refugees",
        "Inuzuka","Kubota","Ghoya","Okura","Kempeitai","Gendarmerie",
        "Naval Landing Party","Nippon",
    ]),
    "Germany and NS-Regime": _kw_pattern([
        "Generalkonsulat","German Consulate","Deutsches Konsulat","Gestapo","Meisinger",
        "SD","Third Reich","Drittes Reich","Nazi","Auswaertiges Amt","Foreign Office",
    ]),
    "Chinese authorities and the population": _kw_pattern([
        "Chinese","Chinesisch","Chinesen","Nanking Government","Nanking Regierung",
        "Guomindang","Chongqing","Mayor of Shanghai","Buergermeister von Shanghai",
        "Chinese Red Cross","Chinesisches Rotes Kreuz",
    ]),
    "Post-war / US authorities": _kw_pattern([
        "US Army","Amerikanische Armee","UNRRA","CNRRA","American Forces",
        "Amerikanische Truppen","US Consulate","US.Konsulat","Repatriation",
        "Rueckfuehrung","Heimschaffung",
    ]),
    # --- Aid organizations ---
    "Local relief committees": _kw_pattern([
        "CAEJR","CFA","Speelman Committee","Speelman Komitee","International Committee",
        "Internationales Komitee","Komor Committee","Kitchen Fund","Kuechenfonds",
        "Gemeindevorstand",
    ]),
    "International relief committees": _kw_pattern([
        "JDC","Joint","AJJDC","HIAS","HICEM","ORT","Quakers","Quaeker",
        "Society of Friends","Red Cross","Rotes Kreuz",
    ]),
    "Ghetto- or collaboration organisations": _kw_pattern([
        "SACRA","Joint Administrative Committee","JAC","Pao Chia","Foreign Pao Chia","Paochia",
    ]),
    # --- Core themes ---
    "Ghettoisation & Spatial Planning": _kw_pattern([
        "Designated Area","Bezeichnetes Gebiet","Ghetto","Hongkew","Hongkou",
        "Passierschein","Permit","Proclamation","Bekanntmachung","Umsiedlung",
        "Relocation","Heim","Ward Road","Pingliang","Chaoufoong",
    ]),
    "Finance, Work & Emergencies": _kw_pattern([
        "Relief","Unterstuetzung","Finanzen","Finance","Subvention","Spenden","Donations",
        "Dole","Arbeitslosigkeit","Unemployed","Bettler","Beggar","Prostitution",
        "Business","Geschaeft","Pacht","Lease","Inflation","CRB dollars",
    ]),
    "Health, Hygiene & Mortality": _kw_pattern([
        "Krankheit","Disease","Epidemie","Epidemic","Typhus","Dysenterie","Dysentery",
        "Malaria","Hospital","Krankenhaus","Ward Road Hospital","Mangelernaehrung",
        "Malnutrition","Nutritional","Friedhof","Cemetery","Suizid","Suicide",
        "Selbstmord","Sterblichkeit","Mortality","Death rate",
    ]),
    "Culture, Religion & the Press": _kw_pattern([
        "Theater","Konzert","Concert","Zeitung","Newspaper","Press","Presse","Schule",
        "School","Kadoorie School","Synagoge","Synagogue","Koscher","Kosher","Kashrut",
        "Sabbat","Sabbath","Kulturkreis",
    ]),
}

MIN_N_FOR_TEST = 30  # below this match count: not formally tested

# Note: despite word boundaries, "ORT" remains ambiguous (collides with
# the German word "Ort/place", not reliably separable even by
# capitalization due to OCR inconsistencies) -- included in the
# "International relief committees" category, but its result should be
# read with caution. A cleaner version would require manually reviewing
# ORT matches, or restricting to quotation-mark context (e.g. "ORT" in
# quotation marks in the original).

# Check match counts first -- determines what is even testable
topic_counts = {}
topic_masks = {}
for label, pattern in TOPIC_KEYWORDS.items():
    mask = articles["text"].str.contains(pattern, case=False, regex=True, na=False)
    topic_masks[label] = mask
    topic_counts[label] = mask.sum()

counts_df = pd.DataFrame({
    "thema": list(topic_counts.keys()),
    "n_treffer": list(topic_counts.values()),
})
counts_df["testbar"] = counts_df["n_treffer"] >= MIN_N_FOR_TEST
counts_df = counts_df.sort_values("n_treffer", ascending=False)
print(counts_df.to_string(index=False))
counts_df.to_csv("topic_keyword_counts.csv", index=False)
print("Table saved as 'topic_keyword_counts.csv'.")


In [ ]:
NORM_COLS = [f"{e}_norm" for e in EMOTIONS]

testable_topics = counts_df[counts_df["testbar"]]["thema"].tolist()
print(f"Formally testable topics (n >= {MIN_N_FOR_TEST}): {testable_topics}")
print(f"NOT testable (too few matches): "
      f"{counts_df[~counts_df['testbar']]['thema'].tolist()}")

rows = []
for label in testable_topics:
    mask = topic_masks[label]
    group_yes = articles[mask]
    group_no = articles[~mask]
    for e_norm in NORM_COLS:
        emotion = e_norm.replace("_norm", "")
        vals_yes = group_yes[e_norm].dropna()
        vals_no = group_no[e_norm].dropna()
        stat, p = stats.mannwhitneyu(vals_yes, vals_no, alternative="two-sided")
        n1, n2 = len(vals_yes), len(vals_no)
        # Rank-biserial correlation as effect size (-1 to 1); positive =
        # the topic group tends to have HIGHER values than the rest of
        # the corpus (sign chosen deliberately to match the
        # median_thema/median_rest columns right next to it -- easier
        # to read).
        effect = (2 * stat) / (n1 * n2) - 1
        rows.append({
            "thema": label, "emotion": emotion, "n_thema": n1, "n_rest": n2,
            "median_thema": vals_yes.median(), "median_rest": vals_no.median(),
            "effektstaerke": effect, "p_roh": p,
        })

topic_emotion_tests = pd.DataFrame(rows)
topic_emotion_tests["p_adj_fdr"] = multipletests(topic_emotion_tests["p_roh"], method="fdr_bh")[1]
topic_emotion_tests["signifikant"] = topic_emotion_tests["p_adj_fdr"] < 0.05
topic_emotion_tests = topic_emotion_tests.sort_values("p_adj_fdr")

print(f"\n{len(topic_emotion_tests)} tests performed, "
      f"{topic_emotion_tests['signifikant'].sum()} significant after FDR correction:\n")
topic_emotion_tests[["thema","emotion","median_thema","median_rest","effektstaerke","p_adj_fdr","signifikant"]]


In [ ]:
# Only significant findings, in plain text
sig = topic_emotion_tests[topic_emotion_tests["signifikant"]]
if sig.empty:
    print("Not a single association survives the multiple-comparison correction.")
    print("Everything that looks like a pattern at first glance is, given")
    print("this corpus size, statistically indistinguishable from chance.")
else:
    for _, row in sig.iterrows():
        richtung = "higher" if row["effektstaerke"] > 0 else "lower"
        print(f"{row['thema']}: {row['emotion']} significantly {richtung} "
              f"(median {row['median_thema']:.3f} vs. {row['median_rest']:.3f} otherwise, "
              f"effect size {row['effektstaerke']:.2f}, p_adj={row['p_adj_fdr']:.3f})")


In [ ]:
# Heatmap of effect sizes for the testable topics -- significance marked
pivot = topic_emotion_tests.pivot(index="thema", columns="emotion", values="effektstaerke")
pivot = pivot[EMOTIONS]  # fixed column order
sig_pivot = topic_emotion_tests.pivot(index="thema", columns="emotion", values="signifikant")[EMOTIONS]

fig, ax = plt.subplots(figsize=(11, 0.6 * len(pivot) + 2))
im = ax.imshow(pivot.values, cmap="RdBu_r", vmin=-0.4, vmax=0.4, aspect="auto")
ax.set_xticks(range(len(EMOTIONS)))
ax.set_xticklabels(EMOTIONS, rotation=45, ha="right")
ax.set_yticks(range(len(pivot)))
ax.set_yticklabels(pivot.index)
for i in range(len(pivot)):
    for j in range(len(EMOTIONS)):
        if sig_pivot.values[i, j]:
            ax.text(j, i, "*", ha="center", va="center", color="black", fontsize=14, fontweight="bold")
plt.colorbar(im, ax=ax, label="Effect Magnitude (rank-biserial correlation)")
ax.set_title("Topic x Emotion: Effect Magnitude (* = significant after FDR correction)")
plt.tight_layout()
plt.savefig("heatmap.png", dpi=150)
plt.show()

topic_emotion_tests.to_csv("topic_emotion_tests.csv", index=False)
counts_df.to_csv("topic_keyword_counts.csv", index=False)
print("Saved: topic_emotion_tests.csv, topic_keyword_counts.csv, heatmap.png")

# The helper columns can now be dropped for good
articles = articles.drop(columns=["_lemmas_pos", "_lang_used"], errors="ignore")


## Next steps

- For year/language comparisons, primarily use the `_norm` columns; for
  single-article ranking and "overall text sentiment", primarily use
  the raw columns.
- Manually spot-check a few articles with very high/low scores --
  does the NRC score match the actual tone of the article?
- `FUZZY_SCORE_CUTOFF` (Section 5b) is an initial, conservative value --
  if needed, manually review a few random correction pairs from
  `fuzzy_de`/`fuzzy_en` to check whether the threshold is appropriate.
- If the TF-IDF word lists (Section 11) seem too coarse, or independent,
  emotion-agnostic topics are desired: genuine topic modelling (LDA/NMF,
  e.g. via `sklearn.decomposition`) would be the next step -- with more
  manual interpretation effort (topics need to be labelled), and is more
  worthwhile for a corpus of this size when experimenting deliberately
  with different numbers of topics.


In [ ]:
def _latex_safe(text):
    """Escapes underscores for use in \\caption/\\label
    (running text, not math mode) -- otherwise pdflatex fails with
    "Missing $ inserted" as soon as the filename (e.g. "top_fear_articles")
    contains an underscore."""
    return text.replace('_', '-')


def _needs_wrapping(series, threshold=40):
    """Detects columns with long running text (e.g. article excerpts) --
    these need a fixed width WITH line wrapping, otherwise the table
    runs as a single, extremely wide line off the edge of the page
    (Overfull \\hbox)."""
    lengths = series.astype(str).str.len()
    return lengths.mean() > threshold


def _add_grid_lines(latex_str):
    """Converts the booktabs style (only \\toprule/\\midrule/\\bottomrule,
    no vertical lines) into a classic grid: \\hline after EVERY row
    (header and data rows); vertical lines are added separately via
    column_format (see below, "|r|r|...|")."""
    latex_str = latex_str.replace(r'\toprule', r'\hline')
    latex_str = latex_str.replace(r'\midrule', r'\hline')
    latex_str = latex_str.replace(r'\bottomrule', r'\hline')

    lines = latex_str.split('\n')
    out = []
    for i, line in enumerate(lines):
        out.append(line)
        stripped = line.strip()
        # Only genuine table rows (header or data rows: contain "&"
        # and end in "\\") additionally get an \hline afterwards.
        if stripped.endswith(r'\\') and '&' in stripped:
            next_stripped = lines[i + 1].strip() if i + 1 < len(lines) else ''
            if next_stripped != r'\hline':  # avoid producing a duplicate \hline
                out.append(r'\hline')
    return '\n'.join(out)


def process_all_csvs(input_dir, output_dir, text_col_width='6cm', drop_columns=None,
                      skip_files=None):
    """drop_columns: list of column names that (if present) are removed
    from EVERY CSV before LaTeX generation -- e.g. "source_file", which
    has the same value in almost every row anyway and only costs
    unnecessary width. Names are matched case-insensitively AND
    regardless of whether "_" or "-" was used in the original. For CSVs
    that do not have the column at all, nothing happens (no error).

    skip_files: filenames (without path, with or without ".csv") that
    are SKIPPED ENTIRELY -- e.g. for raw-data CSVs with very many
    columns (such as "articles_with_sentiment.csv" with 20+ emotion
    columns), which fundamentally cannot be squeezed onto a printed
    page in a meaningful way, no matter how narrow the individual
    columns are made. For such files: either produce a separate,
    curated CSV with only the relevant columns (e.g. via pandas:
    df[["pdf-page","year","lang","Fear","text"]]) and convert that
    separately, or forgo a LaTeX table entirely and instead only
    include the corresponding charts/heatmaps.
    """
    drop_columns = drop_columns or []
    drop_normalized = {c.lower().replace('_', '-') for c in drop_columns}
    skip_files = skip_files or []
    skip_normalized = {os.path.splitext(f)[0].lower() for f in skip_files}

    os.makedirs(output_dir, exist_ok=True)

    for csv_file in glob.glob(os.path.join(input_dir, '*.csv')):
        base_check = os.path.splitext(os.path.basename(csv_file))[0].lower()
        if base_check in skip_normalized:
            print(f"Skipped (too many columns for print): {csv_file}")
            continue

        df = pd.read_csv(csv_file)

        # Remove unwanted columns before anything else
        cols_to_drop = [c for c in df.columns
                         if c.lower().replace('_', '-') in drop_normalized]
        if cols_to_drop:
            df = df.drop(columns=cols_to_drop)

        # Replace underscores in (remaining) column names with hyphens
        df.columns = [str(col).replace('_', '-') for col in df.columns]

        # Determine column format dynamically: numeric columns
        # right-aligned, long text columns as a fixed, wrapping width
        # (prevents the overfull-hbox problem), everything else
        # left-aligned as usual. Vertical bars "|" between EVERY column
        # AND at the start/end.
        col_format = []
        for col in df.columns:
            if pd.api.types.is_numeric_dtype(df[col]):
                col_format.append('r')
            elif not pd.api.types.is_numeric_dtype(df[col]) and _needs_wrapping(df[col]):
                col_format.append(f'p{{{text_col_width}}}')
            else:
                col_format.append('l')
        column_format = '|' + '|'.join(col_format) + '|'

        base_name = os.path.splitext(os.path.basename(csv_file))[0]
        tex_filename = f"{base_name}_table.tex"
        safe_name = _latex_safe(base_name)  # for caption/label

        latex_table = df.to_latex(
            index=False,
            float_format='%.3f',
            caption=f'Results from {safe_name}',
            label=f'tab:{safe_name}',
            escape=True,          # escape special characters in cell contents
            column_format=column_format,  # fixed column widths + vertical lines
            longtable=True,       # allows the table to span multiple pages
                                   # (otherwise, with long text columns/many
                                   # rows, the table no longer fits on one
                                   # page -> requires the longtable package!)
        )
        latex_table = _add_grid_lines(latex_table)  # \hline after every row

        with open(os.path.join(output_dir, tex_filename), 'w', encoding='utf-8') as f:
            f.write(latex_table)

        print(f"Processed {csv_file} -> {tex_filename}")


# Usage
process_all_csvs(
    'CSV', 'LatexTables',
    drop_columns=['source_file'],
    skip_files=['articles_with_sentiment'],  # raw data -- too many columns for print
)
